In [ ]:
!pip install -q --upgrade pip
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 80.2 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 121.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 39.6 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 102.5 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 113.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 100.6 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 43.0 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [ ]:
%%bash
set -euo pipefail

echo "=== Disk usage ==="
df -h .

echo
echo "=== Python packages check ==="
python - <<'PY'
import sys
import torch
try:
    import torchaudio
    print("torchaudio:", torchaudio.__version__)
except Exception as e:
    print("torchaudio import FAILED:", e)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "cuda version:", torch.version.cuda)
PY

# Путь для скачивания
export DATA_ROOT=/kaggle/working/librispeech_data
echo "DATA_ROOT will be: $DATA_ROOT"

# Создаем папку если её нет
mkdir -p "$DATA_ROOT"

# Удаляем частичные файлы от предыдущих неудачных загрузок (если есть)
echo "Removing leftover .partial files if present..."
find "$DATA_ROOT" -maxdepth 1 -type f -name "*.partial" -print -delete || true

# Параметры обучения

export USE_TORCHAUDIO=1
export DOWNLOAD=1   # уже скачали, ставим 0 чтобы не пытаться заново
export LIBRI_SUBSET=train-clean-100
export BATCH_SIZE=1
export STEPS=200    # на CPU лучше поменьше
export SAMPLE_RATE=16000
export HOP_LENGTH=160
export LR=1e-3
export DEVICE=cpu   # <- обязательный момент: принудительно CPU

python train.py

=== Disk usage ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  388K   20G   1% /kaggle/working

=== Python packages check ===
torchaudio: 2.6.0+cu124
torch: 2.6.0+cu124 cuda: False cuda version: 12.4
DATA_ROOT will be: /kaggle/working/librispeech_data
Removing leftover .partial files if present...
[INFO] device: cpu, data: /kaggle/working/librispeech_data, use_torchaudio: True
[INFO] Starting one-batch training: batch_size=1, steps=200, lr=0.001
step 1/200 loss=14.766177
step 10/200 loss=3.156086
step 20/200 loss=3.036641
step 30/200 loss=2.878977
step 40/200 loss=2.858060
step 50/200 loss=2.835730
step 60/200 loss=2.801522
step 70/200 loss=2.763645
step 80/200 loss=2.705637
step 90/200 loss=2.599429
step 100/200 loss=2.431096
step 110/200 loss=2.214348
step 120/200 loss=1.957556
step 130/200 loss=1.660230
step 140/200 loss=1.316831
step 150/200 loss=0.963229
step 160/200 loss=0.666840
step 170/200 loss=0.446041
step 180/200 loss=0.290729
step 190/200 loss=0.

100%|██████████| 5.95G/5.95G [03:51<00:00, 27.6MB/s] 


In [ ]:
import torchaudio

DATA_ROOT = "/kaggle/working/librispeech_data"

torchaudio.datasets.LIBRISPEECH(DATA_ROOT, url="test-clean", download=True)


In [ ]:
# eval_with_tools_metrics.py
import torch
import torchaudio
from torchaudio.datasets import LIBRISPEECH
from src.models.baseline_model import SampleCTCModel
from src.utils.tokenizer import CharTokenizer as Tokenizer
from tools.metrics import batch_metrics, compute_wer, compute_cer  # твои реализации

# === Настройки ===
DATA_ROOT = "/kaggle/working/librispeech_data"
SUBSET = "test-clean"   # dev-clean / test-clean
CHECKPOINT = "checkpoint.pth"
SAMPLE_RATE = 16000
HOP_LENGTH = 160
DEVICE = "cpu"

dataset = LIBRISPEECH(DATA_ROOT, url=SUBSET, download=True)

# === Модель + токенайзер ===
tokenizer = Tokenizer()
model = SampleCTCModel(num_classes=tokenizer.vocab_size,
                       sample_rate=SAMPLE_RATE,
                       hop_length=HOP_LENGTH).to(DEVICE)

# === Загрузка чекпойнта ===
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
if isinstance(ckpt, dict) and "state_dict" in ckpt:
    state = ckpt["state_dict"]
else:
    state = ckpt
model.load_state_dict(state)
model.eval()

refs = []
hyps = []
per_utt = []

N = 50  # сколько примеров прогнать
with torch.no_grad():
    for i, (waveform, sample_rate, transcript, *_ ) in enumerate(dataset):
        if i >= N:
            break

        if sample_rate != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)(waveform)

        inp = waveform.to(DEVICE).unsqueeze(0)  # [1, C, T]

        logits = model(inp)  # [B, T_out, C]
        preds = torch.argmax(logits, dim=-1)  # [B, T_out]
        pred_indices = preds[0].cpu().tolist()

        hyp_text = tokenizer.decode(pred_indices).strip().lower()
        ref_text = transcript.strip().lower()

        refs.append(ref_text)
        hyps.append(hyp_text)

        w = compute_wer(ref_text, hyp_text)
        c = compute_cer(ref_text, hyp_text)
        per_utt.append((i, ref_text, hyp_text, w, c))

        if i < 5:
            print(f"[{i}] REF: {ref_text}")
            print(f"[{i}] HYP: {hyp_text}")
            print(f"[{i}] WER={w:.3f} CER={c:.3f}")
            print("-" * 40)

# === Средние метрики ===
avg_wer, avg_cer, count = batch_metrics(refs, hyps, normalize=True, skip_empty_refs=True)
print(f"Evaluated {count} utterances. Avg WER={avg_wer:.4f}, Avg CER={avg_cer:.4f}")

100%|██████████| 331M/331M [00:12<00:00, 26.8MB/s] 


[0] REF: he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flour fattened sauce
[0] HYP: ho  cireeto o ontco o etc w nh ere  a  o n etc e e heoont n  e tchec  g cll s
[0] WER=1.000 CER=0.722
----------------------------------------
[1] REF: stuff it into you his belly counselled him
[1] HYP: aalt nati eethirint coheon  hn
[1] WER=1.000 CER=0.714
----------------------------------------
[2] REF: after early nightfall the yellow lamps would light up here and there the squalid quarter of the brothels
[2] HYP: etaceonen no  onon  nhen ner  oe in
[2] WER=1.000 CER=0.798
----------------------------------------
[3] REF: hello bertie any good in your mind
[3] HYP: ee n
[3] WER=1.000 CER=0.882
----------------------------------------
[4] REF: number ten fresh nelly is waiting on you good night husband
[4] HYP: me s nre eeo went
[4] WER=1.000 CER=0.797
----------------------------------------
Evaluated 50 ut